# Human-Written Text Dataset Exploration & Cleaning

## Overview

### Final Human Dataset Composition
- **News Articles** – Formal financial & world news (wire-style)
- **Aeon Essays** – Long-form thoughtful essays (narrative / philosophical)
- **Yelp Reviews** – Casual, personal, opinionated restaurant reviews

**Target:** ~600 high-quality human samples (balanced across the three sources)

Formatted: `text, label, generation_type, source`

### Cleaning Process Summary

**1. News Articles**
- Loaded `articles.csv`
- Basic UTF-8 encoding fixes and whitespace normalization
- Removed wire service tags `(AFP)` and dash headings
- Aggressive ASCII encoding cleanup for OCR artifacts
- Fixed missing spaces after punctuation
- Removed or corrected garbled fragments
- Filtered to 70–750 words

**2. Aeon Essays** (Replacement for CommonLit)
- Loaded `essays.csv`
- Kept only top 10 categories (Stories & Literature, History, etc.)
- Basic text cleaning (encoding + whitespace)
- Chunked long essays (~3400 words avg.) into 120–650 word segments
- Applied final length + quality filter

**3. Yelp Reviews**
- Basic text cleaning (encoding + whitespace)
- Filtered to 70–750 words

**Master Dataset Saved to:**
`../data/human_dataset.csv`

## Kaggle links


**essays.csv:** https://www.kaggle.com/datasets/mannacharya/aeon-essays-dataset

**articles.csv:** https://www.kaggle.com/datasets/asad1m9a9h6mood/news-articles

**yelp.csv:** https://www.kaggle.com/datasets/vivekhn/yelp-reviews

In [1]:
# check directory
import os
os.getcwd()

'/Users/nikkileali/Downloads/FAU/Masters/ArtificalIntelligence/AI_Text_Detection/notebooks'

## Helpers

In [2]:
import pandas as pd
import re

In [3]:
# load df
articles = pd.read_csv("../data/raw_kaggle/articles.csv")
essays = pd.read_csv("../data/raw_kaggle/essays.csv")
yelp = pd.read_csv("../data/raw_kaggle/yelp.csv")

In [4]:
def count_words(text):
    return len(str(text).split())

def count_sentences(text):
    return len(re.findall(r'[.!?]+', str(text)))

In [5]:
def clean_text_base(text):
    text = str(text)
    
    # encoding fix
    text = text.encode("utf-8", "ignore").decode("utf-8")
    
    # fix missing space after punctuation
    text = re.sub(r'([a-zA-Z])\.([A-Z])', r'\1. \2', text)
    
    # normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [6]:
# function for cleaning news articles specifcally
def clean_news(text):
    text = clean_text_base(text)
    
    # fix encoding more aggressively
    text = text.encode("ascii", "ignore").decode()
    
    # remove wire tags
    text = text.replace("(AFP)", "")
    
    # remove dash headings
    text = re.sub(r'-.*?-', ' ', text)
    
    return text.strip()

In [7]:
def chunk_text(text, max_words=480, min_words=120):
    if not isinstance(text, str):
        return []
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        if len(chunk.split()) >= min_words:
            chunks.append(chunk)
    return chunks

In [8]:
def final_filter(text):
    words = text.split()
    if len(words) < 60 or len(words) > 900:  # upper limit
        return False
    
    # stronger artifact check
    if re.search(r'\[\d+\]|Q\d+|Name Stage Name|\(noun\)|\(verb\)|\(adjective\)', text, re.I):
        return False
    
    return True

## Articles Load + Clean

In [9]:
# see articles
print(articles.shape)
print(articles.columns)

articles.head()

(2692, 4)
Index(['Article', 'Date', 'Heading', 'NewsType'], dtype='str')


,Article,Date,Heading,NewsType
0,KARACHI: The Sindh government has decided to b...,1/1/2015,sindh govt decides to cut public transport far...,business
1,HONG KONG: Asian markets started 2015 on an up...,1/2/2015,asia stocks up in new year trad,business
2,HONG KONG: Hong Kong shares opened 0.66 perce...,1/5/2015,hong kong stocks open 0.66 percent lower,business
3,HONG KONG: Asian markets tumbled Tuesday follo...,1/6/2015,asian stocks sink euro near nine year,business
4,NEW YORK: US oil prices Monday slipped below $...,1/6/2015,us oil prices slip below 50 a barr,business


In [10]:
# missing values
print("Articles missing:")
print(articles.isna().sum())

Articles missing:
Article     0
Date        0
Heading     0
NewsType    0
dtype: int64


In [11]:
# check word/sentence counts

# add text stats
articles["word_count"] = articles["Article"].apply(count_words)
articles["sentence_count"] = articles["Article"].apply(count_sentences)

# view distributions
print("Articles word count:")
print(articles["word_count"].describe())

Articles word count:
count    2692.000000
mean      291.035290
std       173.705607
min        32.000000
25%       165.750000
50%       263.000000
75%       381.000000
max      3051.000000
Name: word_count, dtype: float64


In [12]:
# article news types
print(articles["NewsType"].value_counts())

NewsType
sports      1408
business    1284
Name: count, dtype: int64


In [13]:
# clean text formatting
articles["Article"] = articles["Article"].apply(clean_news)

### Final Articles DF

In [14]:
# copy df for cleaning
articles_clean = articles.copy()

# save original df index
articles_clean["source_id"] = "articles_" + articles_clean.index.astype(str)

# rename column to text
articles_clean = articles_clean.rename(columns={"Article": "text"})

# filter by word/sentenec counts
    # keep between 120 and 600 words, and at least 4 sentences
articles_clean = articles_clean[
    (articles_clean["word_count"] >= 120) &
    (articles_clean["word_count"] <= 600) &
    (articles_clean["sentence_count"] >= 4)
]

# metadata
articles_clean["source"] = "articles.csv"
articles_clean["label"] = "human"
articles_clean["generation_type"] = "human"

# final columns
articles_clean = articles_clean[
    ["source_id", "source", "text", "label", "generation_type"]
].copy()

print(articles_clean.shape)
articles_clean.head()

(2180, 5)


,source_id,source,text,label,generation_type
3,articles_3,articles.csv,HONG KONG: Asian markets tumbled Tuesday follo...,human,human
5,articles_5,articles.csv,New York: Oil prices tumbled Tuesday to fresh ...,human,human
6,articles_6,articles.csv,KARACHI: Strong bulls on Friday pulled the ben...,human,human
7,articles_7,articles.csv,"Singapore: Oil fell further in Asia Monday, wi...",human,human
9,articles_9,articles.csv,SYDNEY: Oil prices fell 1 percent on Wednesday...,human,human


## Essays Load + CLean

In [15]:
# see commonlit
print(essays.shape)
print(essays.columns)

essays.head()

(2235, 6)
Index(['title', 'description', 'essay', 'authors', 'source_url',
       'thumbnail_url'],
      dtype='str')


,title,description,essay,authors,source_url,thumbnail_url
0,Space exploration,When self-replicating craft bring life to the ...,"Some time late this century, someone will push...",Jay Olson,https://aeon.co//essays/cosmic-expansion-is-a-...,https://images.aeonmedia.co/images/9239658f-b9...
1,History of science,"To the detriment of the public, scientists and...",Would boycotting Russian scientists be an effe...,Lorraine Daston & Peter Harrison,https://aeon.co//essays/science-and-history-ca...,https://images.aeonmedia.co/images/7e9ea9e3-03...
2,Religion,"Once a centre of Afghan culture, Sufism seems ...",My introduction into the world of Afghanistan’...,Annika Schmeding,https://aeon.co//essays/sufi-transitions-betwe...,https://images.aeonmedia.co/images/957fb6c9-40...
3,Thinkers and theories,The intrepid logician Kurt Gödel believed in t...,"As the foremost logician of the 20th century, ...",Alexander T Englert,https://aeon.co//essays/kurt-godel-his-mother-...,https://images.aeonmedia.co/images/cbe24f46-98...
4,Thinkers and theories,"For Rachel Bespaloff, philosophy was a sensual...",Shortly after Rachel Bespaloff’s suicide in 19...,Isabel Jacobs,https://aeon.co//essays/for-rachel-bespaloff-p...,https://images.aeonmedia.co/images/536e31b1-dc...


In [16]:
print("\nEssays missing:")
print(essays.isna().sum())


Essays missing:
title            0
description      0
essay            0
authors          0
source_url       0
thumbnail_url    0
dtype: int64


In [17]:
# check word/sentence counts

# add text stats
essays["word_count"] = essays["essay"].apply(count_words)
essays["sentence_count"] = essays["essay"].apply(count_sentences)

# view distributions
print("essays word count:")
print(essays["word_count"].describe())

essays word count:
count     2235.000000
mean      3422.634452
std        918.560347
min         60.000000
25%       2882.000000
50%       3331.000000
75%       3846.500000
max      10300.000000
Name: word_count, dtype: float64


In [18]:
# top 10 categories
print(essays['title'].value_counts().head(10))

title
Stories and literature              73
History                             71
Thinkers and theories               70
Ethics                              69
Biology                             62
History of ideas                    55
Human rights and justice            45
Cognition and intelligence          42
Consciousness and altered states    40
Art                                 39
Name: count, dtype: int64


In [19]:
# basic cleaning
essays["essay"] = essays["essay"].apply(clean_text_base)

### Final Commonlit DF

In [21]:
# copy df for cleaning
essays_clean = essays.copy()

# save original df index
essays_clean["source_id"] = "essays_" + essays_clean.index.astype(str)

# rename text
essays_clean = essays_clean.rename(columns={"essay": "text"})

# only keep top categories
essays_clean = essays_clean[
    essays_clean["title"].isin([
    "Stories and literature", "History", "Thinkers and theories", "Ethics", 
    "Biology", "History of ideas", "Human rights and justice", 
    "Cognition and intelligence", "Consciousness and altered state", "Art"
])
]

# chunk long text
chunked_rows = []
for _, row in essays_clean.iterrows():
    chunks = chunk_text(row["text"])
    for i, chunk in enumerate(chunks):
        new_row = row.copy()
        new_row["text"] = chunk
        new_row["source_id"] = f"{row['source_id']}_chunk{i}"
        chunked_rows.append(new_row)

essays_clean = pd.DataFrame(chunked_rows)

# metadata
essays_clean["source"] = "essays_texts.csv"
essays_clean["label"] = "human"
essays_clean["generation_type"] = "human"


# final columns
essays_clean = essays_clean[
    ["source_id", "source", "text", "label", "generation_type"]
].copy()

print(essays_clean.shape)
essays_clean.head()

(3822, 5)


,source_id,source,text,label,generation_type
3,essays_3_chunk0,essays_texts.csv,"As the foremost logician of the 20th century, ...",human,human
3,essays_3_chunk1,essays_texts.csv,in which he actively worked out views and expe...,human,human
3,essays_3_chunk2,essays_texts.csv,fully expresses this potential in a lifetime. ...,human,human
3,essays_3_chunk3,essays_texts.csv,to assume that the world is rationally organis...,human,human
3,essays_3_chunk4,essays_texts.csv,of inference. Hence any consistent system will...,human,human


## Yelp Clean + Load

In [22]:
# see yelp
print(yelp.shape)
print(yelp.columns)

yelp.head()

(10000, 10)
Index(['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id',
       'cool', 'useful', 'funny'],
      dtype='str')


,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,1,2,0
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0


In [23]:
print("\nYelp missing:")
print(yelp.isna().sum())


Yelp missing:
business_id    0
date           0
review_id      0
stars          0
text           0
type           0
user_id        0
cool           0
useful         0
funny          0
dtype: int64


In [24]:
# check word/sentence counts

# add text stats
yelp["word_count"] = yelp["text"].apply(count_words)
yelp["sentence_count"] = yelp["text"].apply(count_sentences)

# view distributions
print("yelp word count:")
print(yelp["word_count"].describe())

yelp word count:
count    10000.000000
mean       131.039600
std        113.584114
min          1.000000
25%         54.000000
50%        101.000000
75%        173.000000
max        945.000000
Name: word_count, dtype: float64


In [25]:
# yelp star ratings
print(yelp["stars"].value_counts())

stars
4    3526
5    3337
3    1461
2     927
1     749
Name: count, dtype: int64


In [26]:
# clean text formatting
yelp["text"] = yelp["text"].apply(clean_text_base)

### Final Yelp DF

In [27]:
# copy df for cleaning
yelp_clean = yelp.copy()

# save original df index
yelp_clean["source_id"] = "yelp_" + yelp_clean.index.astype(str)

# filter by word/sentenec counts
    # keep between 120 and 600 words, and at least 4 sentences
yelp_clean = yelp_clean[
    (yelp_clean["word_count"] >= 120) &
    (yelp_clean["word_count"] <= 600) &
    (yelp_clean["sentence_count"] >= 4)
]

# metadata
yelp_clean["source"] = "yelp.csv"
yelp_clean["label"] = "human"
yelp_clean["generation_type"] = "human"


# final columns
yelp_clean = yelp_clean[
    ["source_id", "source", "text", "label", "generation_type"]
].copy()

print(yelp_clean.shape)
yelp_clean.head()

(4116, 5)


,source_id,source,text,label,generation_type
0,yelp_0,yelp.csv,My wife took me here on my birthday for breakf...,human,human
1,yelp_1,yelp.csv,I have no idea why some people give bad review...,human,human
5,yelp_5,yelp.csv,"Quiessence is, simply put, beautiful. Full win...",human,human
6,yelp_6,yelp.csv,Drop what you're doing and drive here. After I...,human,human
14,yelp_14,yelp.csv,I'm 2 weeks new to Phoenix. I looked up Irish ...,human,human


## Final Datasets Check

In [34]:
# check & remove duplicates
print("Articles duplicates:", articles_clean["text"].duplicated().sum())
print("Essays duplicates:", essays_clean["text"].duplicated().sum())
print("Yelp duplicates:", yelp_clean["text"].duplicated().sum())

articles_clean = articles_clean.drop_duplicates(subset=["text"])
essays_clean = essays_clean.drop_duplicates(subset=["text"])
yelp_clean = yelp_clean.drop_duplicates(subset=["text"])

Articles duplicates: 0
Essays duplicates: 0
Yelp duplicates: 0


In [35]:
articles_clean = articles_clean[articles_clean["text"].apply(final_filter)]
essays_clean = essays_clean[essays_clean["text"].apply(final_filter)]
yelp_clean = yelp_clean[yelp_clean["text"].apply(final_filter)]

In [36]:
# check shapes after cleaning
print("Articles:", articles_clean.shape)
print("Essays:", essays_clean.shape)
print("Yelp:", yelp_clean.shape)

Articles: (2034, 5)
Essays: (3818, 5)
Yelp: (4113, 5)


In [37]:
# check columns after cleaning
print(articles_clean.columns)
print(essays_clean.columns)
print(yelp_clean.columns)

Index(['source_id', 'source', 'text', 'label', 'generation_type'], dtype='str')
Index(['source_id', 'source', 'text', 'label', 'generation_type'], dtype='str')
Index(['source_id', 'source', 'text', 'label', 'generation_type'], dtype='str')


In [39]:
# check content after cleaning
print(articles_clean.iloc[0]["text"])
print("-----")

print(essays_clean.iloc[0]["text"])
print("-----")

print(yelp_clean.iloc[0]["text"])

HONG KONG: Asian markets tumbled Tuesday following painful losses in New York and Europe while the euro sat near nine wing Syriza party. Markets fear the party will roll back austerity measures required under the IMF  Oil below $50 a barrel -The Dow dived 1.86 percent, the S&P 500 fell 1.83 percent and the Nasdaq lost 1.57 percent. In currency trade the euro sank to $1.1864 Monday, its lowest level since March 2006. On Tuesday morning the single currency recovered slightly buying $1.1943.The euro was meanwhile at 142.58 yen against 142.74 yen in US trade and well down from the 144.58 yen Friday. Adding to downward pressure is increased speculation that the European Central Bank will buy eurozone government bonds to counter deflation risks. The dollar was at 119.40 yen early Tuesday, compared with 119.61 in New York Monday and also well down from 120.46 yen Friday. Oil prices were marginally up Tuesday after slipping below $50 for the first time in more than five years in New York. US b

## Master Dataset Creation

In [40]:
# generation targets 

TARGET_ARTICLES = 200
TARGET_ESSAYS = 200
TARGET_YELP = 200

In [41]:
# random sample articles based on target amounts

articles_sample = articles_clean.sample(min(TARGET_ARTICLES, len(articles_clean)), random_state=42)
essays_sample = essays_clean.sample(min(TARGET_ESSAYS, len(essays_clean)), random_state=42)
yelp_sample = yelp_clean.sample(min(TARGET_YELP, len(yelp_clean)), random_state=42)

In [42]:
# merge 3 samples into one dataset

human_dataset = pd.concat(
    [articles_sample, essays_sample, yelp_sample],
    ignore_index=True
)

# shuffle
human_dataset = human_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

In [43]:
# check final dataset
print(human_dataset.shape)
print(human_dataset["source"].value_counts())

(600, 5)
source
articles.csv        200
yelp.csv            200
essays_texts.csv    200
Name: count, dtype: int64


In [44]:
# save final dataset
human_dataset.to_csv("../data/human_dataset.csv", index=False)